# 90-Minute Lecture Notebook: Time-Series Analytics and ML for BTS Datasets

**Datasets:**
- DB28 — Air Carrier Traffic Statistics (monthly)
- DB1B — Origin & Destination Survey (quarterly)
- ASQP — Airline Service Quality Performance (monthly)

This notebook:
- Introduces the four layers of analytics (Descriptive, Diagnostic, Predictive, Prescriptive)
- Demonstrates a simple N-BEATS neural-network model for forecasting
- Provides 20 questions (15 single-dataset + 5 cross-dataset) with techniques and representative code snippets


In [ ]:
!pip install pandas numpy matplotlib darts[u] statsmodels --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.dataprocessing.transformers import Scaler
from darts.models import NBEATSModel
import statsmodels.formula.api as smf

plt.rcParams['figure.figsize'] = (10,4)

## Demo 1: DB28 — Monthly Passenger Forecast (Descriptive → Predictive → Prescriptive)

In [ ]:
# Replace with your real DB28 extract
db28 = pd.read_csv('/content/db28_sample.csv')  # expects Year, Month, Passengers, Seats
db28['Date'] = pd.to_datetime(db28[['Year','Month']].assign(Day=1))
db28 = db28.sort_values('Date')

# Descriptive: plot passengers
plt.plot(db28['Date'], db28['Passengers'])
plt.title('DB28 – Monthly Passengers')
plt.xlabel('Date'); plt.ylabel('Passengers')
plt.show()

# N-BEATS forecast
series = TimeSeries.from_dataframe(db28, 'Date', 'Passengers')
scaler = Scaler()
series_scaled = scaler.fit_transform(series)

train, test = series_scaled[:-12], series_scaled[-12:]

model_db28 = NBEATSModel(input_chunk_length=24, output_chunk_length=12, n_epochs=100, random_state=0)
model_db28.fit(train)
forecast_db28 = model_db28.predict(12)

series_scaled.plot(label='Actual')
forecast_db28.plot(label='Forecast')
plt.legend(); plt.show()

# Simple prescriptive rule
growth = (forecast_db28.values()[-1] - train.values()[-1]) / train.values()[-1]
if growth > 0.10:
    print('Recommendation: Increase capacity on key markets.')
else:
    print('Recommendation: Maintain or rebalance capacity.')


## Demo 2: DB1B — Quarterly Fare Forecast

In [ ]:
# Replace with your real DB1B quarterly summary
db1b = pd.read_csv('/content/db1b_quarterly.csv')  # expects Quarter, AvgFare, Passengers, Distance, Competitors
db1b['Date'] = pd.to_datetime(db1b['Quarter'])
db1b = db1b.sort_values('Date')

series_db1b = TimeSeries.from_dataframe(db1b, 'Date', 'AvgFare')
train_db1b, test_db1b = series_db1b[:-4], series_db1b[-4:]

model_db1b = NBEATSModel(input_chunk_length=8, output_chunk_length=4, n_epochs=150, random_state=0)
model_db1b.fit(train_db1b)
forecast_db1b = model_db1b.predict(4)

series_db1b.plot(label='Actual')
forecast_db1b.plot(label='Forecast')
plt.legend(); plt.show()


## Demo 3: ASQP — Monthly On-Time Performance Forecast

In [ ]:
# Replace with your aggregated ASQP data
asqp = pd.read_csv('/content/asqp_monthly.csv')  # expects Year, Month, OTP, FlightsPerHour, delay cause columns
asqp['Date'] = pd.to_datetime(asqp[['Year','Month']].assign(Day=1))
asqp = asqp.sort_values('Date')

series_asqp = TimeSeries.from_dataframe(asqp, 'Date', 'OTP')
train_asqp, test_asqp = series_asqp[:-12], series_asqp[-12:]

model_asqp = NBEATSModel(input_chunk_length=18, output_chunk_length=12, n_epochs=150, random_state=0)
model_asqp.fit(train_asqp)
forecast_asqp = model_asqp.predict(12)

series_asqp.plot(label='Actual')
forecast_asqp.plot(label='Forecast')
plt.legend(); plt.show()

if forecast_asqp.values().min() < 0.80:
    print('Prescriptive signal: add schedule padding / buffers to protect OTP.')


## 4. DB28 — Air Carrier Traffic Statistics: 5 Questions + Techniques + Code

### Q1 (Descriptive): How have monthly passenger counts changed over 5 years?
**Technique:** Line plot + rolling mean.
```python
df = db28.copy()
last5 = df[df['Date'] >= df['Date'].max() - pd.DateOffset(years=5)]
plt.plot(last5['Date'], last5['Passengers'], label='Monthly')
last5['Passengers'].rolling(12).mean().plot(label='12-month rolling mean')
plt.legend(); plt.show()
```

### Q2 (Descriptive): Which months show highest/lowest demand?
**Technique:** Group by month-of-year.
```python
df = db28.copy()
df['Month'] = df['Date'].dt.month
monthly_avg = df.groupby('Month')['Passengers'].mean()
monthly_avg.plot(kind='bar'); plt.title('Average passengers by month'); plt.show()
```

### Q3 (Diagnostic): How did competitor entry affect demand?
**Technique:** Pre/post dummy + OLS.
```python
df = db28.copy()
df['PostEntry'] = df['Date'] >= '2022-01-01'
model = smf.ols('Passengers ~ PostEntry', data=df).fit()
print(model.summary())
```

### Q4 (Predictive): Forecast next 12 months of demand (N-BEATS).
```python
series = TimeSeries.from_dataframe(db28, 'Date', 'Passengers')
scaler = Scaler()
series_scaled = scaler.fit_transform(series)
train, test = series_scaled[:-12], series_scaled[-12:]
model_db28 = NBEATSModel(input_chunk_length=24, output_chunk_length=12, n_epochs=150, random_state=0)
model_db28.fit(train)
forecast_db28 = model_db28.predict(12)
```

### Q5 (Prescriptive): Which routes need capacity increases?
**Technique:** Compare forecast to seats.
```python
# Assume db28 has columns [Date, Origin, Dest, Passengers, Seats]
route = db28[db28['Origin'].eq('AAA') & db28['Dest'].eq('BBB')]
series_route = TimeSeries.from_dataframe(route, 'Date', 'Passengers')
model = NBEATSModel(24, 6, n_epochs=150)
model.fit(series_route[:-6])
fc_route = model.predict(6)

seats_future = route['Seats'].iloc[-6:].values.reshape(-1,1)
lf_forecast = fc_route.values() / seats_future
print('Forecast load factors:', lf_forecast.flatten())
```


## 5. DB1B — Origin & Destination Survey: 5 Questions + Techniques + Code

### Q6 (Descriptive): How have average fares changed over 10 years?
```python
df = db1b.copy()
last10 = df[df['Date'] >= df['Date'].max() - pd.DateOffset(years=10)]
plt.plot(last10['Date'], last10['AvgFare'])
plt.title('Average fare over last 10 years'); plt.show()
```

### Q7 (Diagnostic): How does competition affect fare?
```python
model = smf.ols('AvgFare ~ Competitors', data=db1b).fit()
print(model.summary())
```

### Q8 (Diagnostic): How do fares differ by distance band?
```python
df = db1b.copy()
df['Band'] = pd.cut(df['Distance'], bins=[0,500,1500,3000,6000], labels=['Short','Medium','Long','Ultra'])
df.groupby('Band')['AvgFare'].describe()
```

### Q9 (Predictive): Forecast quarterly fares.
```python
series = TimeSeries.from_dataframe(db1b, 'Date', 'AvgFare')
train, test = series[:-4], series[-4:]
model = NBEATSModel(8, 4, n_epochs=200, random_state=0)
model.fit(train)
fc = model.predict(4)
series.plot(label='Actual'); fc.plot(label='Forecast'); plt.legend(); plt.show()
```

### Q10 (Prescriptive): Which markets react most to fare cuts?
```python
# Simple elasticity approximation per market
elasticities = []
for (o,d), g in db1b.groupby(['Origin','Dest']):
    if len(g) > 8:
        m = smf.ols('Passengers ~ AvgFare', data=g).fit()
        elasticities.append((o,d,m.params['AvgFare']))

elasticities[:10]  # most negative coefficients = most sensitive
```


## 6. ASQP — Airline Service Quality Performance: 5 Questions + Techniques + Code

### Q11 (Descriptive): What is the trend in monthly OTP?
```python
df = asqp.copy()
plt.plot(df['Date'], df['OTP'])
plt.title('Monthly On-Time Performance'); plt.show()
```

### Q12 (Diagnostic): Which delay causes dominate?
```python
cause_cols = ['Weather','Carrier','NAS','Security','LateAircraft']
df[cause_cols].sum().plot(kind='bar'); plt.title('Total delay minutes by cause'); plt.show()
```

### Q13 (Diagnostic): How does schedule density affect delay?
```python
model = smf.ols('ArrDelay ~ FlightsPerHour', data=asqp).fit()
print(model.summary())
```

### Q14 (Predictive): Forecast OTP over next 12 months.
```python
series = TimeSeries.from_dataframe(asqp, 'Date', 'OTP')
train, test = series[:-12], series[-12:]
model = NBEATSModel(18, 12, n_epochs=150, random_state=0)
model.fit(train)
fc = model.predict(12)
series.plot(label='Actual'); fc.plot(label='Forecast'); plt.legend(); plt.show()
```

### Q15 (Prescriptive): Where to add buffers?
```python
risk = fc[fc.values() < 0.80]
risk
```


## 7. Cross-Dataset Questions (Combinations): 5 Questions + Techniques + Code

### Q16: How do fares (DB1B) drive demand (DB28)?
```python
merged = db28.merge(db1b, on=['Origin','Dest','Date'], suffixes=('_db28','_db1b'))
model = smf.ols('Passengers ~ AvgFare', data=merged).fit()
print(model.summary())
```

### Q17: How does reliability (ASQP) affect demand (DB28)?
```python
merged = db28.merge(asqp[['Date','OTP']], on='Date')
model = smf.ols('Passengers ~ OTP', data=merged).fit()
print(model.summary())
```

### Q18: Capacity–Fare–Demand alignment.
```python
df = merged.copy()
df['Revenue'] = df['Passengers'] * df['AvgFare']
df[['Passengers','AvgFare','Seats','Revenue']].head()
```

### Q19: Which routes to prioritize for expansion?
```python
from sklearn.cluster import KMeans
features = merged[['Passengers','AvgFare','OTP']].dropna()
kmeans = KMeans(n_clusters=4, random_state=0).fit(features)
features['cluster'] = kmeans.labels_
features.groupby('cluster').mean()
```

### Q20: Scenario analysis with fuel ↑ and OTP ↓.
```python
# Assume merged has FuelPrice
model = smf.ols('Revenue ~ FuelPrice + OTP + AvgFare', data=merged).fit()
print(model.summary())
```